# 03. 태거 + 어절 사전 결합 디코딩
02 에서 학습한 `outputs/tagger_best.pt` 를 그대로 쓰고(재학습 없음), 예측 때 01 의 어절 사전을 더한다.

`score(y) = log P_태거(초) + log P_태거(중) + log P_태거(종) + dict_weight × P_사전(y)`

- 사전: 입력 어절과 모음 골격이 같은 train 어절들을 '난독화 확률(ㄱㄲㅋ 처럼 같은 그룹이면 높음) × 빈도'로 비교
  - 예) 꾜뀌 → 후보 고기·보기·오이… → ㄱ→ㄲ, ㅗ→ㅛ, ㅣ→ㅟ 가 모두 흔한 변환인 **고기** 가 1위
- 사전에 없는 어절이면 보너스 ≈ 0 → 태거 판단 그대로
- `dict_weight 0` = 태거 단독 (02 결과와 동일) → 사전 효과를 바로 비교

**전제: 02 노트북의 본 학습(셀 3)을 먼저 완료.** GPU 런타임 권장 (CPU 도 가능하지만 느림).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/데이콘/난독화
!ls -la outputs/tagger_best.pt

## 1) 검증: dict_weight × dict_oov 탐색 → `outputs/best_tagger_dict.json`
모든 결과는 `outputs/tagger_dict_runs.csv` 에 누적 기록.

In [ ]:
!python tagger_dict.py --mode val

In [ ]:
import pandas as pd
r = pd.read_csv('outputs/tagger_dict_runs.csv')
r.sort_values('f1', ascending=False).head(10)

## 2) 01 / 02 / 03 검증 결과 비교 (같은 val 분할)

In [ ]:
from collections import Counter
res = {}
for name, f in [('01 noisy-channel', 'outputs/val_pred.csv'),
                ('02 tagger', 'outputs/val_pred_tagger.csv'),
                ('03 tagger+dict', 'outputs/val_pred_tagger_dict.csv')]:
    try:
        d = pd.read_csv(f); d['pred'] = d['pred'].fillna('')
        res[name] = d.set_index('ID'); print(f'{name:18s} F1 {d.f1.mean():.4f}  (rows {len(d)})')
    except FileNotFoundError:
        print(f'{name:18s} 파일 없음: {f}')
if '02 tagger' in res and '03 tagger+dict' in res:
    a, b = res['02 tagger'], res['03 tagger+dict']
    ids = a.index.intersection(b.index)
    diff = (b.loc[ids, 'f1'] - a.loc[ids, 'f1'])
    print(f'사전 추가로 좋아진 행 {(diff > 0).sum()} / 나빠진 행 {(diff < 0).sum()} / 같음 {(diff == 0).sum()}')
    for i in diff.sort_values().index[:3].tolist() + diff.sort_values().index[-3:].tolist():
        print(f'\n[{diff[i]:+.3f}] IN  :', b.loc[i, 'input'][:100])
        print('        GT  :', b.loc[i, 'output'][:100])
        print('        02  :', a.loc[i, 'pred'][:100])
        print('        03  :', b.loc[i, 'pred'][:100])

## 3) 오류 분석 (03)

In [ ]:
v = pd.read_csv('outputs/val_pred_tagger_dict.csv')
err = Counter((a, b) for g, p in zip(v.output, v.pred.fillna('')) for a, b in zip(g, p) if a != b)
print('mean F1', v.f1.mean()); print(err.most_common(30))

## 4) test 예측 → `outputs/submission_tagger_dict.csv`
`best_tagger_dict.json` 설정 자동 사용, 사용한 값은 `submission_tagger_dict_params.json` 에 저장.

In [ ]:
!python tagger_dict.py --mode test
pd.read_csv('outputs/submission_tagger_dict.csv').head()